# Tweet Sentiment & Stock Returns — Reusable Data Audit

**Project:** S&P Global & CRISIL Campus Hackathon — RiskPulse AI  
**Purpose:** Reusable audit + ingestion notebook for the Kaggle **Tweet Sentiment's Impact on Stock Returns** dataset.

This notebook is deliberately written as **codebase infrastructure**, not a disposable report. It:

- detects the `full_dataset-release.csv` and `reduced_dataset-release.csv` files inside the supplied ZIP;
- reconstructs the dataset's malformed record structure safely;
- normalizes types and fields into a canonical schema;
- performs data-quality, distribution, duplication, temporal and leakage checks;
- performs a memory-efficient streaming audit of the full dataset;
- defines reusable functions that can later become `src/data/tweets.py`;
- provides a temporal split strategy for downstream sentiment/event experiments.

> **Important:** the raw CSV is not a conventional one-record-per-line CSV. Tweet text can contain physical line breaks, while the metadata for a tweet appears on a subsequent line. A naïve `pandas.read_csv()` therefore misaligns columns. The parser below reconstructs records using the metadata boundary (`DATE` field) before loading them into a DataFrame.


## 1. Dataset provenance

The supplied Kaggle dataset is **Tweet Sentiment's Impact on Stock Returns**. Public metadata describes 862,231 labelled instances in the full dataset and the fields used here: tweet text, stock, date, price, 1/2/3/7-day returns, volume, 10/30-day volatility and two pre-existing sentiment scores. The downloaded archive also contains a reduced file.

For the hackathon repository, retain the original source attribution and state exactly which files were used. The competition rules require that submitted data be public/synthetic and that the source/assumptions be documented.

**Research context:** the associated literature studies relationships between social-media sentiment/activity and subsequent market behavior, but this notebook treats future returns strictly as **offline evaluation targets**, never as live inference inputs.


In [1]:
from pathlib import Path
import os
import re
import io
import csv
import zipfile
import collections
import warnings

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

SEED = 42

# Expected repository layout:
# project/
# ├── data/raw/archive.zip
# └── notebooks/01_tweet_dataset_audit.ipynb
#
# You can override this with:
#   TWEET_DATA_ZIP=/absolute/path/archive.zip

def find_dataset_zip():
    candidates = [
        os.getenv("TWEET_DATA_ZIP"),
        Path("../data/raw/archive.zip"),
        Path("../../data/raw/archive.zip"),
        Path("data/raw/archive.zip"),
        Path("archive.zip"),
    ]
    for candidate in candidates:
        if candidate and Path(candidate).exists():
            return Path(candidate).resolve()
    raise FileNotFoundError(
        "Dataset ZIP not found. Put archive.zip in data/raw/ or set TWEET_DATA_ZIP."
    )

ZIP_PATH = find_dataset_zip()
print("Dataset ZIP:", ZIP_PATH)
print("Size (MB):", round(ZIP_PATH.stat().st_size / 1024**2, 2))


Dataset ZIP: /mnt/data/archive.zip
Size (MB): 57.88


In [2]:
with zipfile.ZipFile(ZIP_PATH) as z:
    members = [(i.filename, i.file_size, i.compress_size) for i in z.infolist()]

pd.DataFrame(
    members,
    columns=["member", "uncompressed_bytes", "compressed_bytes"]
).assign(
    uncompressed_mb=lambda x: x.uncompressed_bytes / 1024**2,
    compressed_mb=lambda x: x.compressed_bytes / 1024**2,
)[["member", "uncompressed_mb", "compressed_mb"]]


,member,uncompressed_mb,compressed_mb
0,full_dataset-release.csv,225.050627,52.385676
1,reduced_dataset-release.csv,22.211125,5.494407


## 2. Robust raw-record parser

The parser deliberately does **not** use `pd.read_csv()` on the raw member.

Observed raw structure:

```text
tweet row
metadata row
tweet row
metadata row
...
```

Some tweets contain embedded physical line breaks. The metadata row contains the date and therefore provides a reliable record boundary.

The full file has no `MENTION` column; the reduced file does. The parser therefore accepts both schemas.


In [3]:
def is_metadata_line(line: str) -> bool:
    """Return True for the metadata line that closes a tweet record.

    Both dataset variants are supported:
      reduced: 14 comma-separated fields after the leading empty/quote field
      full:    13 comma-separated fields after the leading empty/quote field

    The date is always the third comma-separated field.
    """
    if not line:
        return False

    parts = line.split(",")
    if len(parts) not in (13, 14):
        return False

    if parts[0] not in ("", '"'):
        return False

    return bool(re.fullmatch(r"\d{2}/\d{2}/\d{4}", parts[2]))


def iter_reconstructed_records(zip_path, member):
    """Yield reconstructed records from a malformed dataset CSV.

    Yields:
        list[str]: [row_id, tweet, stock, date, ...metadata]
    """
    with zipfile.ZipFile(zip_path) as z, z.open(member) as raw:
        f = io.TextIOWrapper(raw, encoding="utf-8", errors="replace", newline=None)

        # Header is informational; the actual schema is normalized below.
        header = f.readline().rstrip("\n")
        tweet_lines = []

        for line in f:
            line = line.rstrip("\n")

            if is_metadata_line(line):
                if tweet_lines:
                    first = tweet_lines[0]
                    match = re.match(r"^([^,]+),(.*)$", first)

                    if match:
                        row_id = match.group(1)
                        tweet_text = match.group(2)

                        if tweet_text.startswith('"'):
                            tweet_text = tweet_text[1:]

                        if len(tweet_lines) > 1:
                            tweet_text += "\n" + "\n".join(tweet_lines[1:])

                        metadata = line.split(",")[1:]
                        yield [row_id, tweet_text] + metadata

                    tweet_lines = []
            else:
                tweet_lines.append(line)


CANONICAL_COLUMNS = [
    "row_id",
    "tweet",
    "stock",
    "date",
    "last_price",
    "return_1d",
    "return_2d",
    "return_3d",
    "return_7d",
    "px_volume",
    "volatility_10d",
    "volatility_30d",
    "lstm_polarity",
    "textblob_polarity",
    "mention",
]


def load_reconstructed_dataset(zip_path, member, nrows=None):
    """Materialize one dataset member as a normalized DataFrame.

    nrows is useful for quick development/testing.
    """
    rows = []

    for i, row in enumerate(iter_reconstructed_records(zip_path, member)):
        # Full dataset has no MENTION field.
        if len(row) == 14:
            row = row + [None]

        if len(row) != len(CANONICAL_COLUMNS):
            raise ValueError(
                f"Unexpected reconstructed row length={len(row)} at record {i}"
            )

        rows.append(row)

        if nrows is not None and len(rows) >= nrows:
            break

    df = pd.DataFrame(rows, columns=CANONICAL_COLUMNS)
    return normalize_tweet_dataframe(df)


def normalize_tweet_dataframe(df):
    """Normalize types without changing the raw information."""
    df = df.copy()

    df["tweet"] = (
        df["tweet"]
        .astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    for col in ["stock", "mention"]:
        df[col] = df[col].astype("string").str.strip()

    df["date"] = pd.to_datetime(df["date"], dayfirst=True, errors="coerce")

    numeric_cols = [
        "last_price", "return_1d", "return_2d", "return_3d", "return_7d",
        "px_volume", "volatility_10d", "volatility_30d",
        "lstm_polarity", "textblob_polarity",
    ]

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


## 3. Materialize the reduced dataset

The reduced dataset is small enough for interactive development and is the recommended dataset for local feature engineering.

The full dataset should be used for final experiments where practical, but the streaming audit below avoids requiring the full 862k-row file to fit in memory just to inspect it.


In [4]:
REDUCED_MEMBER = "reduced_dataset-release.csv"
FULL_MEMBER = "full_dataset-release.csv"

tweets = load_reconstructed_dataset(ZIP_PATH, REDUCED_MEMBER)

print("Reduced dataset shape:", tweets.shape)
display(tweets.head(5))


Reduced dataset shape: (85176, 15)


,row_id,tweet,stock,date,last_price,return_1d,return_2d,return_3d,return_7d,px_volume,volatility_10d,volatility_30d,lstm_polarity,textblob_polarity,mention
0,0,RT @robertoglezcano: @amazon #Patents Show Flying Warehouses That Send Delivery Drones To Your Door https://t.co/cZw...,Amazon,2017-01-31,823.48,0.008379,0.014924,0.014924,-0.001263,3137196.0,13.447,16.992,1,0.0,@amazon
1,1,"@FAME95FM1 Jamaicans make money with @Payoneer @PayPal, @paxuminc, @ecoPayz and @okpaycom https://t.co/FWzqUrgsqs",PayPal,2017-01-31,39.78,0.002011,0.012318,0.012318,0.054801,9100057.0,18.769,16.099,-1,0.0,@PayPal
2,2,"@CBSi Jamaicans make money with @Payoneer @PayPal, @paxuminc, @ecoPayz and @okpaycom https://t.co/FWzqUqYRyU",PayPal,2017-01-31,39.78,0.002011,0.012318,0.012318,0.054801,9100057.0,18.769,16.099,1,0.0,@PayPal
3,3,"@Hitz92fm Jamaicans make money with @Payoneer @PayPal, @paxuminc, @ecoPayz and @okpaycom https://t.co/FWzqUqYRyU",PayPal,2017-01-31,39.78,0.002011,0.012318,0.012318,0.054801,9100057.0,18.769,16.099,-1,0.0,@PayPal
4,4,RT @loadsofvans: Retweet this post &amp; follow @loadsofvans for a chance to win a £100 @amazon voucher #win #giveaw...,Amazon,2017-01-31,823.48,0.008379,0.014924,0.014924,-0.001263,3137196.0,13.447,16.992,-1,0.8,@amazon


In [5]:
# Basic schema audit
schema = pd.DataFrame({
    "dtype": tweets.dtypes.astype(str),
    "missing": tweets.isna().sum(),
    "missing_pct": (tweets.isna().mean() * 100).round(3),
    "unique": tweets.nunique(dropna=True),
})
display(schema)


,dtype,missing,missing_pct,unique
row_id,object,0,0.000,85176
tweet,string,0,0.000,60003
stock,string,0,0.000,100
date,datetime64[ns],0,0.000,121
last_price,float64,0,0.000,4166
return_1d,float64,0,0.000,4130
return_2d,float64,0,0.000,4974
return_3d,float64,0,0.000,5527
return_7d,float64,0,0.000,4584
px_volume,float64,0,0.000,4523


In [6]:
# Core coverage
coverage = pd.Series({
    "rows": len(tweets),
    "unique_stocks": tweets["stock"].nunique(),
    "unique_dates": tweets["date"].nunique(),
    "date_min": tweets["date"].min(),
    "date_max": tweets["date"].max(),
    "unique_mentions": tweets["mention"].nunique(dropna=True),
    "duplicate_exact_rows": tweets.duplicated().sum(),
    "duplicate_tweet_text": tweets["tweet"].duplicated().sum(),
})
display(coverage.to_frame("value"))


,value
rows,85176
unique_stocks,100
unique_dates,121
date_min,2017-01-31 00:00:00
date_max,2018-10-31 00:00:00
unique_mentions,100
duplicate_exact_rows,0
duplicate_tweet_text,25173


In [7]:
# Stock concentration
stock_counts = tweets["stock"].value_counts()
display(stock_counts.head(20).to_frame("rows"))

stock_share = (stock_counts / len(tweets) * 100).round(2)
display(stock_share.head(20).to_frame("share_pct"))


,rows
stock,
Nike,12888
eBay,12811
Reuters,8193
Netflix,7037
Apple,5008
Amazon,4950
Google,2913
Starbucks,2082
Facebook,2026


,share_pct
stock,
Nike,15.13
eBay,15.04
Reuters,9.62
Netflix,8.26
Apple,5.88
Amazon,5.81
Google,3.42
Starbucks,2.44
Facebook,2.38


In [8]:
# Return / volatility distributions
market_cols = [
    "return_1d", "return_2d", "return_3d", "return_7d",
    "volatility_10d", "volatility_30d",
]
display(
    tweets[market_cols]
    .describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
    .T
)


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
return_1d,85176.0,0.002216,0.017374,-0.146650,-0.037897,-0.022604,-0.004520,0.000000,0.006603,0.032663,0.058573,0.233973
return_2d,85176.0,0.004430,0.022229,-0.173554,-0.052609,-0.025740,-0.006944,0.000835,0.014065,0.036011,0.081369,0.243639
return_3d,85176.0,0.004583,0.025168,-0.177851,-0.055032,-0.035687,-0.007881,0.002762,0.016685,0.040559,0.096282,0.243639
return_7d,85176.0,0.007912,0.038596,-0.204959,-0.073866,-0.046168,-0.014327,0.003306,0.027090,0.068280,0.156930,0.267113
volatility_10d,85176.0,23.871644,14.855819,2.368000,6.103000,8.626000,14.776000,20.517000,26.885000,54.408000,80.447000,124.137000
volatility_30d,85162.0,22.285613,10.430833,4.525000,8.728000,11.527000,16.208000,19.239000,24.216000,41.890000,60.266000,74.355000


In [9]:
# Existing sentiment baselines
sentiment_cols = ["lstm_polarity", "textblob_polarity"]
display(
    tweets[sentiment_cols]
    .describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
    .T
)

print("\nLSTM value counts:")
display(tweets["lstm_polarity"].value_counts(dropna=False).sort_index().to_frame("rows"))

print("\nTextBlob value counts (top 15):")
display(tweets["textblob_polarity"].value_counts(dropna=False).head(15).to_frame("rows"))


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
lstm_polarity,85176.0,0.101883,0.994802,-1.0,-1.0000,-1.000000,-1.0,1.0,1.000000,1.0,1.0,1.0
textblob_polarity,85173.0,0.062713,0.290267,-1.0,-0.7764,-0.466667,0.0,0.0,0.136364,0.6,1.0,1.0



LSTM value counts:


,rows
lstm_polarity,
-1,38249
1,46927



TextBlob value counts (top 15):


,rows
textblob_polarity,
0.000000,45233
0.500000,2862
0.136364,2014
-0.600000,1897
0.200000,1696
0.250000,1454
1.000000,1368
0.100000,1219
0.400000,1105


## 4. Duplicate analysis — do **not** blindly deduplicate

Repeated tweet text is not automatically a duplicate record.

A single tweet can be associated with multiple stocks when it mentions several companies. Also, repeated text can arise from retweets or repeated social posts.

Therefore the canonical dataset should retain row-level records. Any event aggregation later should explicitly define its unit:

- tweet-level;
- stock-day;
- stock-event;
- cross-source event cluster.

The analysis below identifies repeated text and repeated text across stocks rather than deleting it.


In [10]:
tweet_groups = (
    tweets.groupby("tweet", dropna=False)
    .agg(
        occurrences=("tweet", "size"),
        unique_stocks=("stock", "nunique"),
        unique_dates=("date", "nunique"),
    )
    .sort_values("occurrences", ascending=False)
)

print("Unique tweet texts:", len(tweet_groups))
print("Repeated tweet texts:", int((tweet_groups["occurrences"] > 1).sum()))
print(
    "Repeated texts mapped to >1 stock:",
    int((tweet_groups["unique_stocks"] > 1).sum())
)

display(tweet_groups.head(20))


Unique tweet texts: 60003
Repeated tweet texts: 7682
Repeated texts mapped to >1 stock: 942


,occurrences,unique_stocks,unique_dates
tweet,,,
RT @Nike: Don’t ask if your dreams are crazy. Ask if they’re crazy enough. #justdoit https://t.co/Wd5L42egV8,1387,1,18
"RT @Apple: The all-new iPhone XS, iPhone XS Max, iPhone XR and Apple Watch Series 4 are here. #AppleEvent",914,1,10
RT @DevanF91: A thread: reason why I chose to cut @Nike swoosh logo of my socks. https://t.co/3JFmRMgjVH,292,1,7
RT @Apple: Join us September 12 at 10 a.m. PDT to watch the #AppleEvent live on Twitter. Tap ❤️ below and we’ll send you updates on event d…,241,1,4
RT @trvo512: are you fucking kidding me? @McDonalds https://t.co/YgNCoLqJhr,232,1,2
RT @Manal_Cali: Hey @Nike can you sue her for illegally using your trademark and name without permission? https://t.co/XLeHqsZknc,217,1,4
RT @Nike: Just a small-town kid with big-city dreams. #justdoit @KingJames https://t.co/okTQBM8hLI,210,1,5
RT @netflix: they made us an offer we couldn't refuse https://t.co/HLIQj56yFy,198,1,2
RT @netflix: 24 years ago today Rachel Green ran out of her wedding and into Central Perk 💕 ☕️ https://t.co/zSZUNYxPxm,174,1,6


## 5. Full-dataset streaming audit

The full file is about 225 MB uncompressed and contains 862,231 reconstructed records. We can audit it without materializing the complete DataFrame.

This function is reusable in CI/data-validation scripts later.


In [11]:
def stream_dataset_audit(zip_path, member):
    numeric_cols = [
        "last_price", "return_1d", "return_2d", "return_3d", "return_7d",
        "px_volume", "volatility_10d", "volatility_30d",
        "lstm_polarity", "textblob_polarity",
    ]

    n = 0
    stocks = collections.Counter()
    dates = collections.Counter()
    missing = collections.Counter()
    stats = {
        c: {"count": 0, "sum": 0.0, "sumsq": 0.0,
            "min": float("inf"), "max": float("-inf")}
        for c in numeric_cols
    }

    for row in iter_reconstructed_records(zip_path, member):
        n += 1
        stocks[row[2]] += 1
        dates[row[3]] += 1

        for pos, col in zip(range(4, 14), numeric_cols):
            value = row[pos] if pos < len(row) else ""

            try:
                x = float(value)
            except (TypeError, ValueError):
                x = np.nan

            if not np.isfinite(x):
                missing[col] += 1
                continue

            s = stats[col]
            s["count"] += 1
            s["sum"] += x
            s["sumsq"] += x * x
            s["min"] = min(s["min"], x)
            s["max"] = max(s["max"], x)

    summary = []
    for col, s in stats.items():
        mean = s["sum"] / s["count"] if s["count"] else np.nan
        summary.append({
            "column": col,
            "count": s["count"],
            "missing": missing[col],
            "mean": mean,
            "min": s["min"] if s["count"] else np.nan,
            "max": s["max"] if s["count"] else np.nan,
        })

    return {
        "rows": n,
        "unique_stocks": len(stocks),
        "unique_dates": len(dates),
        "min_date": min(dates) if dates else None,
        "max_date": max(dates) if dates else None,
        "stock_counts": stocks,
        "date_counts": dates,
        "numeric_summary": pd.DataFrame(summary),
    }


full_audit = stream_dataset_audit(ZIP_PATH, FULL_MEMBER)

print("Full rows:", full_audit["rows"])
print("Unique stocks:", full_audit["unique_stocks"])
print("Unique dates:", full_audit["unique_dates"])
print("Date range:", full_audit["min_date"], "→", full_audit["max_date"])
display(full_audit["numeric_summary"])


Full rows: 862231
Unique stocks: 101
Unique dates: 121
Date range: 01/02/2017 → 31/10/2018


,column,count,missing,mean,min,max
0,last_price,862231,0,1.630092e+03,3.010000,1.655000e+05
1,return_1d,862231,0,8.767511e-04,-0.146650,2.339725e-01
2,return_2d,862231,0,1.674989e-03,-0.173554,2.436389e-01
3,return_3d,862231,0,1.793072e-03,-0.177851,2.436389e-01
4,return_7d,862231,0,4.141522e-03,-0.204959,2.671133e-01
5,px_volume,862231,0,1.698582e+07,1.000000,3.148332e+08
6,volatility_10d,862231,0,2.490885e+01,0.619000,1.241370e+02
7,volatility_30d,862085,146,2.546607e+01,4.525000,8.768500e+01
8,lstm_polarity,862231,0,1.299060e-01,-1.000000,1.000000e+00
9,textblob_polarity,862227,4,7.331058e-02,-1.000000,1.000000e+00


## 6. Findings that affect our architecture

### Finding 1 — The raw CSV requires a custom ingestion adapter

This is not optional. A normal `pd.read_csv()` misaligns records because tweets contain physical line breaks.

**Implementation consequence:** the reconstruction logic belongs in the data-ingestion layer, not as a one-off notebook hack.

### Finding 2 — Full dataset size

The full archive contains **862,231 reconstructed records**. The reduced file contains **85,176 reconstructed records**.

**Implementation consequence:** use the reduced file during development; run final validation/experiments on the full file where compute permits.

### Finding 3 — Sentiment labels are not independent human ground truth

`LSTM_POLARITY` and `TEXTBLOB_POLARITY` are existing model outputs.

**Implementation consequence:** use them as **baselines/comparison features**, not as gold labels for training FinBERT.

### Finding 4 — Future returns are valid evaluation targets but invalid live features

`1_DAY_RETURN` through `7_DAY_RETURN` occur after the tweet.

**Implementation consequence:** they can only be used in offline evaluation/backtesting. They must never enter the live Risk Engine feature vector.

### Finding 5 — Repeated tweets exist

Exact text repetition is substantial.

**Implementation consequence:** don't globally deduplicate raw tweets. Instead, later build event clustering at the stock/event/source level.

### Finding 6 — Market context is available

Price, volume and 10/30-day volatility are available.

**Implementation consequence:** our impact model can eventually combine NLP signals with market context without leaking future returns.

### Finding 7 — The dataset is highly imbalanced by stock

A relatively small set of companies accounts for a large fraction of records.

**Implementation consequence:** evaluation should report both overall metrics and per-stock / macro-averaged metrics where appropriate.


## 7. Leakage-safe evaluation design

We will use a **temporal split**, not a random train/test split.

Recommended initial design:

```text
2017-01 → 2018-03     TRAIN
2018-04 → 2018-07     VALIDATION
2018-08 → 2018-10     TEST
```

The exact boundaries should be calculated from the observed date range and finalized once the model-training task is defined.

Why:

- random splitting allows near-duplicate/repeated events from adjacent dates to cross partitions;
- future market regimes can leak into training;
- financial NLP evaluation should preserve chronology.

For any experiment that uses future returns as targets, features must be generated only from information available at or before the tweet timestamp/date.


In [12]:
def temporal_split(df, train_end, val_end):
    """Chronological split for leakage-safe experiments."""
    train_end = pd.Timestamp(train_end)
    val_end = pd.Timestamp(val_end)

    train = df[df["date"] <= train_end].copy()
    val = df[(df["date"] > train_end) & (df["date"] <= val_end)].copy()
    test = df[df["date"] > val_end].copy()

    return train, val, test


# Example only — finalize boundaries after the model task is specified.
train, val, test = temporal_split(
    tweets,
    train_end="2018-03-31",
    val_end="2018-07-31",
)

print("Train:", train.shape)
print("Validation:", val.shape)
print("Test:", test.shape)


Train: (23324, 15)
Validation: (690, 15)
Test: (61162, 15)


## 8. Canonical schema for RiskPulse

The raw dataset should not become our application schema directly.

The downstream Risk Engine should operate on a normalized event/message object such as:

```text
message_id
source
source_timestamp
text
entity
entity_id / ticker
market_context
raw_metadata
```

The historical dataset can populate that schema with:

- `source = "historical_tweets"`
- `text = tweet`
- `entity = stock`
- `source_timestamp = date`
- market context = price/volume/volatility

Future GDELT/NewsAPI records should be normalized into the **same interface**.

That gives us one Risk Engine with multiple ingestion adapters.


In [13]:
def to_risk_message_schema(df, source="historical_tweets"):
    """Convert historical tweet records to the common ingestion schema.

    This intentionally excludes future-return columns from the live message object.
    Returns remain available in the historical DataFrame for offline evaluation.
    """
    out = pd.DataFrame({
        "message_id": df["row_id"].astype("string"),
        "source": source,
        "source_timestamp": df["date"],
        "text": df["tweet"],
        "entity": df["stock"],
        "mention": df["mention"],
        "last_price": df["last_price"],
        "px_volume": df["px_volume"],
        "volatility_10d": df["volatility_10d"],
        "volatility_30d": df["volatility_30d"],
    })

    return out


risk_messages = to_risk_message_schema(tweets.head(5))
display(risk_messages)


,message_id,source,source_timestamp,text,entity,mention,last_price,px_volume,volatility_10d,volatility_30d
0,0,historical_tweets,2017-01-31,RT @robertoglezcano: @amazon #Patents Show Flying Warehouses That Send Delivery Drones To Your Door https://t.co/cZw...,Amazon,@amazon,823.48,3137196.0,13.447,16.992
1,1,historical_tweets,2017-01-31,"@FAME95FM1 Jamaicans make money with @Payoneer @PayPal, @paxuminc, @ecoPayz and @okpaycom https://t.co/FWzqUrgsqs",PayPal,@PayPal,39.78,9100057.0,18.769,16.099
2,2,historical_tweets,2017-01-31,"@CBSi Jamaicans make money with @Payoneer @PayPal, @paxuminc, @ecoPayz and @okpaycom https://t.co/FWzqUqYRyU",PayPal,@PayPal,39.78,9100057.0,18.769,16.099
3,3,historical_tweets,2017-01-31,"@Hitz92fm Jamaicans make money with @Payoneer @PayPal, @paxuminc, @ecoPayz and @okpaycom https://t.co/FWzqUqYRyU",PayPal,@PayPal,39.78,9100057.0,18.769,16.099
4,4,historical_tweets,2017-01-31,RT @loadsofvans: Retweet this post &amp; follow @loadsofvans for a chance to win a £100 @amazon voucher #win #giveaw...,Amazon,@amazon,823.48,3137196.0,13.447,16.992


## 9. Optional materialization

Do not commit the original 225 MB full CSV to the application repository unless the competition explicitly requires it and repository limits permit it.

For development, it is reasonable to materialize the reconstructed reduced dataset as Parquet:

```python
out = Path("../data/processed/tweets_reduced.parquet")
out.parent.mkdir(parents=True, exist_ok=True)
tweets.to_parquet(out, index=False)
```

For the final repository, follow the hackathon's data-submission instructions and document the source, license and exact preprocessing procedure.


# 10. Immediate engineering conclusions

We can now move to the Risk Engine with a clear contract.

### Historical social-data adapter

```text
Kaggle ZIP
   ↓
record reconstruction
   ↓
normalization
   ↓
canonical message
```

### Live news adapters

```text
GDELT / NewsAPI
   ↓
canonical message
```

### Shared NLP pipeline

```text
canonical message
   ↓
entity resolution
   ↓
FinBERT sentiment
   ↓
event classification
   ↓
event clustering
   ↓
impact score
   ↓
confidence
```

### Offline evaluation only

```text
risk signal at T
      +
future return T+1/T+7
      ↓
backtest / statistical evaluation
```

This separation is now the foundation for the implementation.
